# Notebook 3b — Cosmic Chronometers

Cosmic chronometers measure $H(z)$ directly from the differential ages of passively evolving galaxies:

$$H(z) = -\frac{1}{1+z}\frac{dz}{dt}$$

The theoretical prediction for flat ΛCDM is simply:

$$H_{th}(z) = 100\, h\, E(z)$$

Unlike SNe, $h$ enters directly — both $\Omega_m$ and $h$ are constrained independently.

The $\chi^2$ is:

$$\chi^2(\Omega_m, h) = \sum_i \frac{\left(H_{th}(z_i) - H_{obs}(z_i)\right)^2}{\sigma_i^2}$$

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from utils import *
from plots import *
from chi2 import chi2_cc

---
## 1. Load the data `[S1]`

In [ ]:
z_cc, H_cc, dH_cc = np.loadtxt('../data/cosmic_chrono.txt', unpack=True, skiprows=1)

print(f"Loaded {len(z_cc)} cosmic chronometer measurements")
print(f"Redshift range: {z_cc.min():g} — {z_cc.max():g}")

---
## 2. Plot data vs theory `[S2]`

In [ ]:
Om = 0.3
h  = 0.7
rho_de, _ = de_model('LCDM', Om)

z_th  = np.linspace(0, z_cc.max() * 1.1, 300)
a_th  = z_to_a(z_th)
H_th  = 100 * h * np.sqrt(E2(a_th, Om, rho_de))

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(z_cc, H_cc, yerr=dH_cc, fmt='o', color='#e34948',
            ms=5, capsize=3, label='cosmic chronometers')
ax.plot(z_th, H_th, color='#2a78d6', lw=2,
        label=f'ΛCDM ($\\Omega_m={Om:g}$, $h={h:g}$)')
ax.set_xlabel(r'redshift $z$')
ax.set_ylabel(r'$H(z)$  [km/s/Mpc]')
ax.set_title('Cosmic chronometers — data vs ΛCDM')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 3. 2D $\chi^2$ scan over $(\Omega_m, h)$ `[S3]`

In [ ]:
Om_arr = np.linspace(0.1, 0.6, 60)
h_arr  = np.linspace(0.5, 0.9, 60)
chi2_grid = np.zeros((len(Om_arr), len(h_arr)))

for i, Om in enumerate(Om_arr):
    rho_de, _ = de_model('LCDM', Om)
    for j, h in enumerate(h_arr):
        chi2_grid[i, j] = chi2_cc(Om, h, rho_de, z_cc, H_cc, dH_cc)

chi2_min = chi2_grid.min()
idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
Om_best = Om_arr[idx[0]]
h_best  = h_arr[idx[1]]

print(f"Best fit Om = {Om_best:g}")
print(f"Best fit h  = {h_best:g}")
print(f"chi2_min    = {chi2_min:g}")
print(f"chi2/dof    = {chi2_min/(len(z_cc)-2):g}")

fig, ax = plt.subplots(figsize=(7, 6))
ct = ax.contour(Om_arr, h_arr, (chi2_grid - chi2_min).T,
                levels=[2.30, 6.17], colors=['#2a78d6', '#e34948'])
ax.clabel(ct, fmt={2.30: r'$1\sigma$', 6.17: r'$2\sigma$'}, fontsize=10)
ax.plot(Om_best, h_best, 'k*', ms=10, label=f'best fit')
ax.set_xlabel(r'$\Omega_m$')
ax.set_ylabel(r'$h$')
ax.set_title(r'Cosmic chronometers — $\chi^2(\Omega_m, h)$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 4. 1D scans `[S4]`

Fix one parameter at its best-fit value and scan over the other.

In [ ]:
# scan over h, fix Om at best fit
rho_de, _ = de_model('LCDM', Om_best)
chi2_h = np.array([chi2_cc(Om_best, h, rho_de, z_cc, H_cc, dH_cc) for h in h_arr])

# scan over Om, fix h at best fit
chi2_Om = np.zeros(len(Om_arr))
for i, Om in enumerate(Om_arr):
    rho_de, _ = de_model('LCDM', Om)
    chi2_Om[i] = chi2_cc(Om, h_best, rho_de, z_cc, H_cc, dH_cc)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(h_arr, chi2_h - chi2_h.min(), color='#2a78d6')
axes[0].axvline(h_best, color='#e34948', ls='--', lw=1.5,
                label=f'$h = {h_best:g}$')
axes[0].axhline(1, color='black', ls=':', lw=1, alpha=0.5)
axes[0].set_xlabel(r'$h$')
axes[0].set_ylabel(r'$\Delta\chi^2$')
axes[0].set_title(r'Fixed $\Omega_m$')
axes[0].legend(fontsize=10)

axes[1].plot(Om_arr, chi2_Om - chi2_Om.min(), color='#2a78d6')
axes[1].axvline(Om_best, color='#e34948', ls='--', lw=1.5,
                label=f'$\\Omega_m = {Om_best:g}$')
axes[1].axhline(1, color='black', ls=':', lw=1, alpha=0.5)
axes[1].set_xlabel(r'$\Omega_m$')
axes[1].set_ylabel(r'$\Delta\chi^2$')
axes[1].set_title(r'Fixed $h$')
axes[1].legend(fontsize=10)

plt.suptitle('Cosmic chronometers — 1D scans')
plt.tight_layout()
plt.show()